# Analyzing saved MoE routing / activation traces

This notebook runs on **CPU**. It does **not** load the language model. It only reads an experiment directory
written by `python -m moe_interp.cli run ...` and calls the same functions the `analyze` and `plot` CLI commands use.

Set `RESULTS` to your experiment directory. If metrics are missing, they are computed from the saved traces.
If the directory does not exist, the notebook stops. It never invents data.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from moe_interp.analysis import AnalysisSettings, Experiment, load_metrics, run_analysis
from moe_interp import plotting
from moe_interp.storage import load_trace

# Point this at a real experiment directory, e.g. results/smoke_test or a Google Drive path.
RESULTS = Path(os.environ.get("MOE_INTERP_RESULTS", "../results/debug"))
assert (RESULTS / "generations.jsonl").exists(), f"No experiment at {RESULTS.resolve()}; run `moe_interp.cli run` first."
print("Using", RESULTS.resolve())

## 1. Metrics
`AnalysisSettings` mirrors the `analyze` CLI flags. Change `alignment` to `common_prefix` for a sanity check: identical prefixes should give overlap ≈ 1.

In [ ]:
settings = AnalysisSettings(reference="clean", control="benign", alignment="common_suffix")
if not (RESULTS / "metrics" / "overlap_summary.csv").exists():
    run_analysis(RESULTS, settings)
tables = load_metrics(RESULTS)
print(sorted(tables))
tables["alignment"]

In [ ]:
# Pairs that could not be aligned are excluded from every metric. Check why:
al = tables["alignment"]
al[~al["ok"]] if not al.empty else al

In [ ]:
tables["generation_summary"]

In [ ]:
ov = tables["overlap_summary"]
ov.pivot(index="layer", columns="condition", values="overlap_mean").round(3) if not ov.empty else ov

## 2. Figures
These are the same functions `make_all_plots` uses. Each returns a matplotlib figure, or `None` if its table is empty.

In [ ]:
conds = list(dict.fromkeys(list(tables["alignment"].get("reference", [])) + list(tables["expert_freq"]["condition"])))
colors = plotting.condition_colors(conds)
for fig in [plotting.plot_overlap(tables.get("overlap_summary"), colors),
            plotting.plot_jsd(tables.get("jsd_summary"), tables.get("jsd_pooled"), colors),
            plotting.plot_activation_similarity(tables.get("activation_similarity_summary"), colors, "cosine")]:
    if fig is not None:
        display(fig); plt.close(fig)

In [ ]:
fig = plotting.plot_expert_freq_heatmaps(tables.get("expert_freq"), conds)
display(fig) if fig is not None else print("no expert_freq table"); plt.close("all")

In [ ]:
fig = plotting.plot_control_delta(tables.get("control_delta_summary"), colors)
display(fig) if fig is not None else print("no control-adjusted deltas (needs reference + control + treatment)"); plt.close("all")

In [ ]:
# Write every figure to <RESULTS>/plots (PNG; set save_pdf=True for PDFs too)
written = plotting.make_all_plots(RESULTS, save_pdf=False)
len(written)

## 3. Working with raw traces
Each trace is plain tensors plus `trace.json` metadata, so custom analyses need no model.

In [ ]:
exp = Experiment.open(RESULTS)
sid = next(iter(exp.samples()))
by_cond = exp.samples()[sid]
traces = {c: exp.load(r) for c, r in by_cond.items()}
t = traces["clean"] if "clean" in traces else next(iter(traces.values()))
print("sample:", sid, "| conditions:", list(traces))
print("topk_ids", tuple(t.topk_ids.shape), "= [layers, prompt tokens, k]")
print("topk_weights", tuple(t.topk_weights.shape), "(dispatched weights; sum per token =",
      round(float(t.topk_weights.sum(-1).mean()), 3), ")")
print("routing source:", t.meta["routing"]["source"], "| norm_topk_prob:", t.meta["routing"]["norm_topk_prob"])
print("activations captured at positions", t.positions[:10], "...", "types:", t.activation_types())

In [ ]:
# Experts chosen for the final prompt token at every layer, side by side across conditions.
rows = {c: [sorted(tr.topk_ids[l, -1].tolist()) for l in range(tr.num_layers)] for c, tr in traces.items()}
pd.DataFrame(rows).rename_axis("layer")

In [ ]:
# Token-aligned comparison by hand: the common suffix, with original indices preserved.
from moe_interp.alignment import align_tokens
from moe_interp.metrics import topk_overlap
if "clean" in traces and len(traces) > 1:
    other = next(c for c in traces if c != "clean")
    al = align_tokens(traces["clean"].input_ids, traces[other].input_ids, "common_suffix")
    print(al.summary())
    ov = topk_overlap(traces["clean"].topk_ids[:, al.positions_a].numpy(), traces[other].topk_ids[:, al.positions_b].numpy())
    print("per-layer mean overlap:", ov.mean(-1).round(3))